# Knox Design Analysis Notebook

Spring 2026

EC552 - Computational Synthetic Biology for Engineers

Team Members Names: Varsha Athreya & Melissa Regalado 

# Notes From James

Depending on number of rules, the rule evaluation algorithm can take 10-30 minutes to run. Rule Evaluations are saved in Neo4j, so you do not
need to rerun the rule evaluation algorithm, call getRuleEvaluation with the name of the evaluation (only 2-5 seconds to retrieve).

Run rule evaluations with:
- ruleEvaluateByGroup
- ruleEvaluateByDesigns


Get rule evaluations with
- getRuleEvaluation


Delete rule evaluations with
- deleteRuleEvaluation


This notebook is to serve as a starting point for doing your analysis.

Feel free to change this notebook in any way you see fit.


# Imports

In [ ]:
import requests
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from sklearn import tree
from sklearn.model_selection import train_test_split

print("imports worked")

# Knox Request Functions

In [23]:
import requests
import pandas as pd
import json

url = 'http://localhost:8080'

def ruleEvaluateByGroup(evalName, groupID, ruleGroupID, labelingMethod="sign"):
    """
    API Request to Knox to run Rule Evaluation Algorithm.

    Returns:
    metrics (list of pandas DataFrames): purity_metrics, designToRule
    """
    response = requests.post(
        url + '/rule/evaluate?' +
        "evaluationName=" + evalName + '&' +
        "designGroupID=" + groupID + '&' +
        "rulesGroupID=" + ruleGroupID + '&' +
        "labelingMethod=" + labelingMethod
    )

    print("Status code:", response.status_code)
    print("Response text:", response.text)

    return processRuleEval(response)


def ruleEvaluateByDesigns(evalName, designIDs, ruleGroupID, designScores, labelingMethod="sign"):
    designSpaceIDs = listToStringList(designIDs)
    designScoresStr = listToStringList(designScores)

    response = requests.post(
        url + '/rule/evaluate?' +
        "evaluationName=" + evalName + '&' +
        "designSpaceIDs=" + designSpaceIDs + '&' +
        "rulesGroupID=" + ruleGroupID + '&' +
        "designScores=" + designScoresStr + '&' +
        "labelingMethod=" + labelingMethod
    )

    return processRuleEval(response)


def getRuleEvaluation(evalName):
    response = requests.get(url + '/rule/getEvaluation?' + "evaluationName=" + evalName)
    return processRuleEval(response)


def deleteRuleEvaluation(evalName):
    response = requests.delete(url + '/rule?' + "evaluationName=" + evalName)

    if not response.text:
        return f'"{evalName}" Successfully Deleted'
    else:
        return response.text


def processRuleEval(response):
    json_data = json.loads(response.text)

    print("Parsed JSON response:")
    print(json_data)

    if "evaluationResults" not in json_data:
        print("Missing 'evaluationResults' in response.")
        return json_data

    if "designToRule" not in json_data:
        print("Missing 'designToRule' in response.")
        return json_data

    purity_metrics_df = pd.DataFrame(json_data["evaluationResults"]).T
    print("purity_metrics_df columns:", purity_metrics_df.columns.tolist())

    designToRule_df = pd.DataFrame(
        json_data["designToRule"],
        index=json_data["designToRule"]["designIDs"]
    )
    cols = designToRule_df.columns.to_list()
    cols.remove("labels")
    cols.remove("scores")
    cols.remove("designIDs")
    cols = ["labels", "scores"] + cols
    designToRule_df = designToRule_df[cols]

    if "scores" in designToRule_df.columns:
        designToRule_df = designToRule_df.sort_values("scores")

    return purity_metrics_df, designToRule_df


def listToStringList(list_input):
    return ",".join(list_input)

# Decision Tree Functions

In [24]:
def exampleTree(X, y, **kwargs):
    # Mess around with different parameters
    
    dt_clf = tree.DecisionTreeClassifier(
        splitter=kwargs.get('splitter', 'best'),
        max_depth=kwargs.get('max_depth', None),
        min_samples_split=kwargs.get('min_samples_split', 400),
        min_samples_leaf=kwargs.get('min_samples_leaf', 200),
        max_features=kwargs.get('max_features', None),
        max_leaf_nodes=kwargs.get('max_leaf_nodes', None)
    )
    
    dt_clf = dt_clf.fit(X, y)

    return dt_clf

## Build more Trees, multiclassification and regression


# Design Analysis

### Run RuleEvaluation

In [ ]:
#deleteRuleEvaluation('test')

In [ ]:
# Use this for (gnn_predicted_scores.csv) designs already have the attached scores

evalName = 'ml_rules_eval_1'
groupID = 'ml_designs_3996'
ruleGroupID = "gg_rules"

labelingMethod = 'median'

purity_metrics_df, designToRule_df = ruleEvaluateByGroup(evalName=evalName, groupID=groupID, ruleGroupID=ruleGroupID, labelingMethod=labelingMethod)



In [ ]:
# Use this for extra credit 1 (true_scores.csv & transformer_predicted_scores.csv) attach new scores to the designs

evalName = "test"
designIDs = [f'outputPrefix_design_{i}' for i in range(1, 3997)]
designScores = [str(j) for j in pd.read_csv(r"C:\Path\to\transformer_predicted_scores.csv")["weight"].to_list()]
ruleGroupID = 'ruleGroup'

labelingMethod = 'median'

purity_metrics_df, designToRule_df = ruleEvaluateByDesigns(evalName, designIDs, ruleGroupID, designScores, labelingMethod)

### View DataFrames

In [ ]:
purity_metrics_df

In [ ]:
designToRule_df

In [ ]:
#saving files to csvs 

purity_metrics_df.to_csv("purity_metrics.csv", index=True)
designToRule_df.to_csv("design_to_rule.csv", index=True)

## Extract Features and Labels from designToRule_df

In [ ]:
feature_names = designToRule_df.columns.to_list()[2:]
X = designToRule_df.iloc[:, 2:].to_numpy(dtype=int)
y_labels = designToRule_df["labels"].to_numpy(dtype=int)
y_scores = designToRule_df["scores"].to_numpy(dtype=float)


## Train - Test Split

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

# Features
feature_cols = [c for c in designToRule_df.columns if c not in ["labels", "scores"]]
X = designToRule_df[feature_cols]

# Targets
y_binary = designToRule_df["labels"]          # binary classification
y_reg = designToRule_df["scores"]             # regression

# Multi-class target using score quantiles
y_multi = pd.qcut(
    designToRule_df["scores"],
    q=4,
    labels=["awful", "poor", "good", "great"],
    duplicates="drop"
)

# 70/30 train-test splits
X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X, y_binary, test_size=0.30, random_state=42
)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.30, random_state=42
)

X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X, y_multi, test_size=0.30, random_state=42
)

## Build Trees

In [ ]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error

# Binary Classification Tree
binary_tree = DecisionTreeClassifier(max_depth=5, random_state=42)
binary_tree.fit(X_train_bin, y_train_bin)
y_pred_bin = binary_tree.predict(X_test_bin)
print("Binary Classification Accuracy:", accuracy_score(y_test_bin, y_pred_bin))

# Regression Tree
regression_tree = DecisionTreeRegressor(max_depth=5, random_state=42)
regression_tree.fit(X_train_reg, y_train_reg)
y_pred_reg = regression_tree.predict(X_test_reg)
print("Regression MAE:", mean_absolute_error(y_test_reg, y_pred_reg))
print("Regression MSE:", mean_squared_error(y_test_reg, y_pred_reg))

# Multi-Classification Tree
multi_tree = DecisionTreeClassifier(max_depth=5, random_state=42)
multi_tree.fit(X_train_multi, y_train_multi)
y_pred_multi = multi_tree.predict(X_test_multi)
print("Multi-Classification Accuracy:", accuracy_score(y_test_multi, y_pred_multi))

## Plot Trees

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

# Binary Classification Tree
plt.figure(figsize=(24, 12))
plot_tree(
    binary_tree,
    feature_names=feature_cols,
    class_names=["0", "1"],
    filled=True,
    fontsize=8
)
plt.title("Binary Classification Tree")
plt.show()

# Regression Tree
plt.figure(figsize=(24, 12))
plot_tree(
    regression_tree,
    feature_names=feature_cols,
    filled=True,
    fontsize=8
)
plt.title("Regression Tree")
plt.show()

# Multi-Classification Tree
plt.figure(figsize=(24, 12))
plot_tree(
    multi_tree,
    feature_names=feature_cols,
    class_names=[str(c) for c in y_multi.cat.categories],
    filled=True,
    fontsize=8
)
plt.title("Multi-Classification (Binned) Tree")
plt.show()

## Save Trees

In [ ]:
# Save Binary Classification Tree
plt.figure(figsize=(24, 12))
plot_tree(
    binary_tree,
    feature_names=feature_cols,
    class_names=["0", "1"],
    filled=True,
    fontsize=8
)
plt.title("Binary Classification Tree")
plt.savefig("binary_classification_tree.png", bbox_inches="tight")
plt.show()

# Save Regression Tree
plt.figure(figsize=(24, 12))
plot_tree(
    regression_tree,
    feature_names=feature_cols,
    filled=True,
    fontsize=8
)
plt.title("Regression Tree")
plt.savefig("regression_tree.png", bbox_inches="tight")
plt.show()

# Save Multi-Classification Tree
plt.figure(figsize=(24, 12))
plot_tree(
    multi_tree,
    feature_names=feature_cols,
    class_names=[str(c) for c in y_multi.cat.categories],
    filled=True,
    fontsize=8
)
plt.title("Multi-Classification (Binned) Tree")
plt.savefig("multi_classification_tree.png", bbox_inches="tight")
plt.show()

## Save Purity Metrics and DesignToRule DF to CSV

In [ ]:
purity_metrics_df.to_csv("purity_metrics_df.csv", index=True)
designToRule_df.to_csv("designToRule_df.csv", index=True)

print("Saved purity_metrics_df.csv")
print("Saved designToRule_df.csv")

# Deliverables